# Auxiliary ASR Diagonal Attention Evaluation

In [ ]:

# check available CUDA devices
import torch
devices = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        device_name = torch.cuda.get_device_name(i)
        devices.append({
            'type': 'CUDA',
            'available': True,
            'name': device_name,
            'index': i
        })
else:
    devices.append({'type': 'CUDA', 'available': False, 'name': 'N/A'})
devices


In [ ]:

# change folder into the root of the ASR project
import os

if not os.path.isdir('Configs'):
    %cd ../

!pwd


In [ ]:
# import packages and helper utilities
import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
import torch
import yaml

from meldataset import build_dataloader
from models import ASRCNN
from token_map import build_token_map_from_data

def cfg_get_nested(cfg: Dict, path, default=None, sep: str = '.'):
    """Get a nested value from a dict using a list of keys or a dot-separated string."""
    if isinstance(path, str):
        keys = path.split(sep) if path else []
    else:
        keys = path
    cur = cfg
    for key in keys:
        if isinstance(cur, dict) and key in cur:
            cur = cur[key]
        else:
            return default
    return cur

def resolve_ssl_frontend_config(config: Dict) -> Dict:
    ssl_cfg = cfg_get_nested(config, 'ssl_frontend', {}) or {}
    if not isinstance(ssl_cfg, dict):
        return {'enabled': False}
    resolved = dict(ssl_cfg)
    if not resolved.get('enabled', False):
        return {'enabled': False}

    default_models = {
        'wav2vec2': 'facebook/wav2vec2-base',
        'hubert': 'facebook/hubert-base-ls960',
        'wavlm': 'microsoft/wavlm-base',
        'xlsr': 'facebook/wav2vec2-large-xlsr-53',
    }
    selected_architecture = resolved.get('architecture')
    selected_model_name = resolved.get('pretrained_model_name')

    for candidate in ('wav2vec2', 'hubert', 'wavlm', 'xlsr'):
        candidate_cfg = resolved.get(candidate)
        if isinstance(candidate_cfg, dict) and candidate_cfg.get('enabled'):
            selected_architecture = candidate
            candidate_model = candidate_cfg.get('pretrained_model_name') or candidate_cfg.get('model_name')
            if candidate_model:
                selected_model_name = candidate_model
            break

    if selected_architecture is None:
        selected_architecture = 'wav2vec2'
    if not selected_model_name:
        selected_model_name = default_models.get(selected_architecture)

    if not selected_model_name:
        return {'enabled': False}

    resolved['architecture'] = selected_architecture
    resolved['pretrained_model_name'] = selected_model_name
    resolved['enabled'] = True
    return resolved

def get_preprocess_sample_rate(config: Dict) -> int:
    if 'preprocess_params' in config:
        key = 'preprocess_params'
    elif 'preprocess_parasm' in config:
        key = 'preprocess_parasm'
    else:
        return 24000
    return int(cfg_get_nested(config, f'{key}.sr', 24000))

def load_token_map_from_config(config: Dict) -> Dict[str, int]:
    token_src = config.get('phoneme_maps_path')
    if not token_src:
        return build_token_map_from_data(
            config.get('train_data'),
            config.get('val_data'),
            config.get('ood_data'),
            apply_asr_tokenizer=True,
        )
    if isinstance(token_src, dict):
        return token_src
    csv = pd.read_csv(token_src, header=None).values
    return {str(word): int(index) for word, index in csv}

def unpack_ctc_outputs(model_output):
    if isinstance(model_output, dict):
        logits = (
            model_output.get('logits_ctc')
            or model_output.get('ctc_logit')
            or model_output.get('logits')
            or model_output.get('ctc')
        )
        encoder_lengths = model_output.get('encoder_lengths')
    elif isinstance(model_output, (tuple, list)):
        logits = model_output[0] if len(model_output) > 0 else None
        encoder_lengths = None
        for item in model_output[1:]:
            if torch.is_tensor(item) and item.dim() == 1:
                encoder_lengths = item
                break
    else:
        logits = model_output
        encoder_lengths = None
    if logits is None:
        raise ValueError('Could not extract CTC logits from model output.')
    return logits, encoder_lengths

def prepare_batch_for_model(batch, use_ssl_frontend, device):
    if use_ssl_frontend:
        texts, text_lens, mels, mel_lens, wave_tensor, wave_lengths = batch
        features = wave_tensor.to(device)
        feature_lengths = wave_lengths.to(device)
        mel_inputs = mels
        mel_lengths = mel_lens
    else:
        texts, text_lens, mels, mel_lens = batch
        features = mels.to(device)
        feature_lengths = mel_lens.to(device)
        mel_inputs = mels
        mel_lengths = mel_lens
    return {
        'texts': texts,
        'text_lengths': text_lens,
        'mel_inputs': mel_inputs,
        'mel_lengths': mel_lengths,
        'features': features,
        'feature_lengths': feature_lengths,
        'feature_lengths_cpu': feature_lengths.detach().to(torch.long).cpu(),
    }

def load_asr_model_from_config(config: Dict, model_path: str, device: torch.device) -> Tuple[ASRCNN, Dict[str, int], Dict]:
    token_map = load_token_map_from_config(config)

    model_params = cfg_get_nested(config, 'model_params', {
        'input_dim': 80,
        'hidden_dim': 256,
        'n_token': len(token_map),
        'token_embedding_dim': 512,
        'n_layers': 5,
        'location_kernel_size': 31,
    })
    if 'n_token' not in model_params:
        model_params['n_token'] = len(token_map)

    ssl_frontend_config = resolve_ssl_frontend_config(config)
    model = ASRCNN(**model_params, ssl_frontend_config=ssl_frontend_config)
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    state_dict = checkpoint.get('model', checkpoint)
    try:
        model.load_state_dict(state_dict)
    except RuntimeError:
        sanitized_state = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(sanitized_state)

    model.to(device)
    model.eval()
    return model, token_map, ssl_frontend_config

def parse_metadata_csv(csv_path: str) -> List[List[str]]:
    path = Path(csv_path)
    if not path.is_file():
        raise FileNotFoundError(f"Validation CSV '{csv_path}' not found.")

    triples: List[List[str]] = []
    with path.open('r', encoding='utf-8') as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            parts = line.split('|')
            if len(parts) < 2:
                continue
            wav_path = parts[0]
            if len(parts) == 2:
                text = parts[1]
                speaker = ''
            else:
                text = '|'.join(parts[1:-1])
                speaker = parts[-1]
            triples.append([wav_path, text, speaker])
    if not triples:
        raise ValueError(f"No entries found in '{csv_path}'.")
    return triples

def resolve_dataset_params(config: Dict) -> Dict:
    preprocess_key = 'preprocess_params' if 'preprocess_params' in config else 'preprocess_parasm'
    base = {
        'dict_path': cfg_get_nested(config, 'phoneme_maps_path', 'Data/word_index_dict.txt'),
        'sr': cfg_get_nested(config, f'{preprocess_key}.sr', 24000),
        'spect_params': cfg_get_nested(config, f'{preprocess_key}.spect_params', {
            'n_fft': 1024,
            'win_length': 1024,
            'hop_length': 300,
        }),
        'mel_params': cfg_get_nested(config, f'{preprocess_key}.mel_params', {'n_mels': 80}),
    }

    dataset_overrides = cfg_get_nested(config, 'dataset_params', {})
    if isinstance(dataset_overrides, dict):
        for key in ('dict_path', 'sr', 'spect_params', 'mel_params'):
            if key in dataset_overrides:
                base[key] = dataset_overrides[key]
        spec_augment = dataset_overrides.get('spec_augment')
        if spec_augment:
            base['spec_augment_params'] = spec_augment
    return base

def build_dev_dataloader_from_config(config: Dict, device: torch.device, ssl_enabled: bool, batch_size: Optional[int] = None, num_workers: Optional[int] = None):
    val_csv_path = cfg_get_nested(config, 'val_data', None)
    if val_csv_path is None:
        raise ValueError("Validation CSV path ('val_data') not found in the config.")

    path_list = parse_metadata_csv(val_csv_path)
    dataset_cfg = resolve_dataset_params(config)

    if batch_size is None:
        batch_size = min(8, int(cfg_get_nested(config, 'batch_size', 8)))
    dataloader_params = cfg_get_nested(config, 'dataloader_params', {})
    if num_workers is None:
        num_workers = int(dataloader_params.get('val_num_workers', 2))

    collate_config = {'return_wave': bool(ssl_enabled)}
    loader = build_dataloader(
        path_list,
        validation=True,
        batch_size=batch_size,
        num_workers=num_workers,
        device='cuda' if device.type == 'cuda' else 'cpu',
        dataset_config=dataset_cfg,
        collate_config=collate_config,
    )
    return loader, path_list


In [ ]:
# load model, config, and validation dataloader
checkpoint_dir = 'Checkpoint'
config_path = 'Checkpoint/config.yml'

if not os.path.isdir(checkpoint_dir):
    raise FileNotFoundError(f"Checkpoint directory '{checkpoint_dir}' not found.")

ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.startswith('epoch_') and f.endswith('.pth')]
if not ckpt_files:
    raise FileNotFoundError(f"No checkpoint files found in '{checkpoint_dir}'.")

ckpt_files = sorted(ckpt_files, key=lambda x: int(x.split('_')[-1].split('.')[0]))
model_path = os.path.join(checkpoint_dir, ckpt_files[-1])

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print(f'model --> {model_path}')
print(f'config --> {config_path}')
phoneme_source = config.get('phoneme_maps_path', 'built from dataset')
print(f'dictionary --> {phoneme_source}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, token_map, ssl_frontend_config = load_asr_model_from_config(config, model_path, device)
SSL_FRONTEND_ENABLED = bool(ssl_frontend_config.get('enabled', False))
INPUT_SAMPLE_RATE = get_preprocess_sample_rate(config)
if SSL_FRONTEND_ENABLED:
    arch = ssl_frontend_config.get('architecture', 'unknown')
    pretrained_name = ssl_frontend_config.get('pretrained_model_name', 'unknown')
    print(f"SSL frontend enabled: {arch} ({pretrained_name})")
else:
    print('SSL frontend disabled; using mel-spectrogram inputs.')

dev_loader, val_entries = build_dev_dataloader_from_config(config, device, SSL_FRONTEND_ENABLED)
print(f'Validation dataset size: {len(val_entries)} samples')
print(f'Batch size --> {dev_loader.batch_size}')


In [ ]:
# attention alignment diagnostics
import numpy as np

@torch.no_grad()
def diagonal_attention_score(model: ASRCNN, dev_loader, device: Optional[torch.device] = None, band: float = 0.1, max_batches: Optional[int] = None):
    """Compute the average diagonal attention concentration score.

    Args:
        model: ASR model returning alignment matrices when teacher-forced.
        dev_loader: validation dataloader yielding model inputs.
        device: device for computation; defaults to model parameters' device.
        band: allowable deviation from the diagonal (0-1 range).
        max_batches: limit the number of batches to evaluate (useful for quick checks).

    Returns:
        mean_score: average ratio of attention mass inside the diagonal band.
        scores: list of per-utterance scores.
    """
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    diag_scores: List[float] = []

    for batch_idx, batch in enumerate(dev_loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        batch_data = prepare_batch_for_model(batch, SSL_FRONTEND_ENABLED, device)
        texts = batch_data['texts'].to(device)
        text_lens = batch_data['text_lengths'].to(torch.long)
        features = batch_data['features']
        feature_lengths = batch_data['feature_lengths']

        model_kwargs = {
            'text_input': texts,
            'input_lengths': feature_lengths,
        }
        if SSL_FRONTEND_ENABLED:
            model_kwargs['sampling_rate'] = INPUT_SAMPLE_RATE

        outputs = model(features, **model_kwargs)
        if isinstance(outputs, dict):
            attn = outputs.get('attn') or outputs.get('attention') or outputs.get('alignments')
            encoder_lengths = outputs.get('encoder_lengths')
            if attn is None:
                raise KeyError('Model output dictionary does not contain attention matrices.')
        elif isinstance(outputs, (tuple, list)):
            if len(outputs) < 3:
                raise ValueError('Model forward output does not include attention tensors.')
            attn = outputs[2]
            encoder_lengths = outputs[3] if len(outputs) > 3 else None
        else:
            raise TypeError('Unsupported model output type for attention extraction.')

        attn = attn.detach()
        time_axis = attn.size(-1)
        output_axis = attn.size(1)

        if encoder_lengths is None:
            encoder_lengths_cpu = batch_data['feature_lengths_cpu']
        else:
            encoder_lengths_cpu = encoder_lengths.detach().to(torch.long).cpu()

        text_lens_list = text_lens.cpu().tolist()
        mel_lens_list = encoder_lengths_cpu.tolist()

        for b in range(attn.size(0)):
            To = min(int(text_lens_list[b]), output_axis)
            Te = min(int(mel_lens_list[b]), time_axis)
            if To <= 1 or Te <= 1:
                continue

            a = attn[b, :To, :Te]
            total_mass = a.sum()
            if torch.isclose(total_mass, torch.tensor(0.0, device=a.device)):
                continue

            t = torch.arange(To, device=a.device, dtype=torch.float32).unsqueeze(1)
            e = torch.arange(Te, device=a.device, dtype=torch.float32).unsqueeze(0)
            t = t / (To - 1) if To > 1 else torch.zeros_like(t)
            e = e / (Te - 1) if Te > 1 else torch.zeros_like(e)
            diag = t - e
            mask = (diag.abs() <= band).to(a.dtype)

            score = (a * mask).sum() / total_mass.clamp_min(1e-8)
            diag_scores.append(float(score))

    mean_score = float(np.mean(diag_scores)) if diag_scores else float('nan')
    return mean_score, diag_scores


### Diagonal Attention Score
- scores > 0.6–0.7 are usually good; trending higher across training is a healthy sign.

In [ ]:
# evaluate the diagonal attention score
mean_score, scores = diagonal_attention_score(model, dev_loader, device=device, band=0.1, max_batches=None)
print(f'Diagonal attention score: {mean_score:.4f}')
print(f'Evaluated {len(scores)} alignments')

if scores:
    scores_array = np.array(scores)
    print('Score statistics:')
    print(f'  min:  {scores_array.min():.4f}')
    print(f'  25%:  {np.percentile(scores_array, 25):.4f}')
    print(f'  median: {np.median(scores_array):.4f}')
    print(f'  75%:  {np.percentile(scores_array, 75):.4f}')
    print(f'  max:  {scores_array.max():.4f}')
